# Multi-chain on-chain regime analysis

Thin front-end over the `onchain_regime` package (same code path as `python -m onchain_regime run`).

- Data: Litview (LTC) and Bitview (BTC) REST APIs; Dogecoin is stubbed in `assets.py` (`TODO(doge)`).
- Rule `current`: days where MVRV **and** supply-in-profit were both at or below today's values.
- Rule `pctl:<n>`: both at or below the n-th point-in-time (expanding) percentile.

In [ ]:
import os
from pathlib import Path

# Run from the repo root so the shared data/raw cache is reused.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

import pandas as pd
from IPython.display import display

from onchain_regime.pipeline import run_analysis
from onchain_regime.plots import plot_mvrv_overlay, plot_price_vs_realized, plot_relative_mvrv, plot_sip_overlay

pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:.3f}".format)
%matplotlib inline

In [ ]:
ASSETS = ["ltc", "btc"]  # add "doge" once enabled in assets.py
RULE = "current"         # or "pctl:10"
REFRESH = False           # True to re-download instead of using data/raw

res = run_analysis(ASSETS, rule=RULE, refresh=REFRESH)
print("Rule:", res.rule.label)

## 1. How rare are current readings?

In [ ]:
display(res.current.T)
for k, r in res.assets.items():
    print(k.upper())
    display(r.percentiles)

## 2. Forward returns after matching episodes

In [ ]:
for k, r in res.assets.items():
    print(f"=== {k.upper()} episodes ===")
    display(r.returns["episodes"])
    print("From episode starts")
    display(r.returns["episode_summary"])
    print("From all matching days (overlapping windows)")
    display(r.returns["matching_day_summary"])
    print("Unconditional baseline")
    display(r.returns["baseline_summary"])

## 3. Relative valuation and regime timing

In [ ]:
if res.relative_summary is not None:
    display(res.relative_summary)
    display(res.timing)
    display(res.overlap)

## Plots

In [ ]:
plot_mvrv_overlay(res.frames, res.episodes);
plot_sip_overlay(res.frames, res.episodes);
for rel in res.relative.values():
    plot_relative_mvrv(rel, res.episodes);
plot_price_vs_realized(res.frames, res.episodes);

## Monthly view

In [ ]:
for k, r in res.assets.items():
    print(k.upper())
    display(r.monthly.tail(12))